In [57]:
using Pkg
Pkg.add("Combinatorics")
Pkg.add("Distributions")
Pkg.add("Bits")
Pkg.add("Random")
Pkg.add("Parsers")
Pkg.add("LinearAlgebra")
Pkg.add("DataFrames", io=devnull)
Pkg.add("Plots", io=devnull)
Pkg.add("Measures", io=devnull)

using LinearAlgebra
using Distributions
using Random
using Bits
using Parsers
using DataFrames
using Plots, Measures

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.8/Project.toml`
  No Changes to `~/.julia/environments/v1.8/Manifest.toml`


1. Generate Random starting points.
2. Get individual from starting points.
3. Get starting points from individual.
4. Check if starting points are within acceptable range.

In [67]:
# Random Starting point generator k - length of motif, t is number of sequences
function RandomGenerator(DnaSequenceLength, k, t)
    randomStartPoints = Vector{Int16}(undef, 0)
    for i in range(1, t)
        push!(randomStartPoints, ceil(Int, rand()*(DnaSequenceLength - k + 1)))
    end
    return randomStartPoints
end

# represent each starting point as 16 bits and concat them together and return
function getIndividual(randomStartingPoints)
    result = ""
    for i in randomStartingPoints
        temp = string(bits(Int16(i)))
        temp = replace(temp, "<" => "")
        temp = replace(temp, ">" => "")
        temp =replace(temp, " " => "")
        result *= temp
    end
    return result
end

# get numericIndividual from the individual
function getNumericIndividual(regularIndividual)
    result = []
    numIndividuals = Int(length(regularIndividual)/16)
    for i in range(0, numIndividuals-1)
        temp = regularIndividual[Int(16*i+1):Int(16*i+16)]
        push!(result, parse(Int64, temp; base=2))
    end
    return result
    
end


# check if starting points are valid in numeric Individual
function validateIndividual(regularIndividual, DnaSequences, k)
    numericIndividual = getNumericIndividual(regularIndividual)
    for i in range(1, length(numericIndividual))
        if numericIndividual[i] > length(DnaSequences[1]) - k + 1
            return false
        end
    end
    return true
end

validateIndividual (generic function with 1 method)

1. Get Fitness score of a set of motifs for a set of starting points.

In [68]:
# pseudo count for each nucelotide is 1 and sum of pseudo counts = 4(1) = 4
function getFitnessScore(motifs, k, t)
    ScorerDict = Dict([
        ("A", zeros(k)),
        ("C", zeros(k)),
        ("G", zeros(k)),
        ("T", zeros(k))
    ])
    #println(motifs)
    S = 0
    # A, C, G, T order 
    bgCount = Dict([
        ("A", 0),
        ("C", 0),
        ("G", 0),
        ("T", 0)            
    ])
    for col in range(1, k)
        for row in range(1, t)
            ScorerDict[string(motifs[row][col])][col] += 1
            S += 1
            bgCount[string(motifs[row][col])] += 1
        end
    end

    fitness = 0
    for col in range(1, k)
        informationContent = 0
        for nuc in ["A", "C", "T", "G"]
            freqObs = (ScorerDict[nuc][col] + 1)/(t - 1 + 4)
            freqBg = (bgCount[nuc] + 1)/(S + 4)
            informationContent += freqObs*(log(2, (freqObs/freqBg)))
        end
        fitness += informationContent
    end

    return fitness
end

getFitnessScore (generic function with 1 method)

1. perform crossover.
2. perform mutation.
3. perform selection - returns indices of selected individuals - roulette wheel selection based on their fitness scores

In [60]:
# get 2 bit strings and output 2 offspring bit strings
function performCrossover(parent1, parent2)
    
    crossPosition = Int(ceil(rand()*length(parent1)))
    #println(crossPosition)
    
    offspring1 = parent1[1:crossPosition]*parent2[crossPosition+1:end]
    offspring2 = parent1[crossPosition+1:end]*parent2[1:crossPosition]
    
    return offspring1, offspring2
end

# get 1 bit string and output 1 mutated bit string
function performMutation(regularIndividual, mutationRate, mutationProb, k)
    
    for i in range(1, mutationRate)
        mutPosition = Int(ceil(rand()*(length(regularIndividual) - k + 1)))
        #println(mutPosition)
        currProb = rand()
        #println(currProb)
        temp = '0'
        if(currProb <= mutationProb)
            # perform mutation on that random bit
            if(regularIndividual[mutPosition] == '1')
                temp = '0'
            else
                temp = '1'
            end
            if mutPosition == 1
                return temp*regularIndividual[2:end]
            else
                return regularIndividual[1:mutPosition-1]*temp*regularIndividual[mutPosition+1:end]
            end
        end 
    end
    
    return regularIndividual
end

# roulette wheel selection operation 
function performSelection(numSelections, fitnessScores)
    
    normalizedFitnessScores = normalize(fitnessScores)
    individualIndices = []
    for i in range(1, length(fitnessScores))
        push!(individualIndices, i)
    end
    selectedIndices = wsample(individualIndices, normalizedFitnessScores, numSelections, replace = false)
    
    return selectedIndices
end

performSelection (generic function with 1 method)

In [69]:
function consensusMotif(bestMotifs)
    
    consensusDict = Dict([
        ("A", zeros(length(bestMotifs[1]))),
        ("C", zeros(length(bestMotifs[1]))),
        ("G", zeros(length(bestMotifs[1]))),
        ("T", zeros(length(bestMotifs[1])))
    ])
    
    for i in range(1, length(bestMotifs[1]))
        for motif in bestMotifs
            consensusDict[string(motif[i:i])][i] += 1
        end
    end
        
    consensusMotif = ""
    for i in range(1, length(bestMotifs[1]))
        maxCount = 0
        currNuc = ""
        if consensusDict["A"][i] > maxCount
            maxCount = consensusDict["A"][i]
            currNuc = "A"
        end
        if consensusDict["C"][i] > maxCount
            maxCount = consensusDict["C"][i]
            currNuc = "C"
        end
        if consensusDict["G"][i] > maxCount
            maxCount = consensusDict["G"][i]
            currNuc = "G"
        end
        if consensusDict["T"][i] > maxCount
            maxCount = consensusDict["T"][i]
            currNuc = "T"
        end
        consensusMotif *= currNuc
    end
    return consensusMotif
end

consensusMotif (generic function with 1 method)

In [70]:
# using shift width to evaluate all "possible" candidates in proximity
# motif width k and number of sequences t 
function evaluateCandidate(DnaSequences, numericIndividual, shiftWidth, k, t)
    maxFitness = 0
    bestWidth = 0
    motifs = []
    #println(numericIndividual) 
    #println(DnaSequences)
    for i in range(-shiftWidth, shiftWidth)
        #print(i)
        # get motifs based on starting positions in individual
        for j in range(1, t)
            if numericIndividual[j] + i < 1 || numericIndividual[j] + i > length(DnaSequences[1]) - k + 1
                motifs = []
                break
            end
            push!(motifs, DnaSequences[j][numericIndividual[j] + i:numericIndividual[j] + i + k - 1])
        end
        if length(motifs) != 0 
            currFitness = getFitnessScore(motifs, k, t)
            #println(i, currFitness, motifs)
            if(currFitness > maxFitness)
                maxFitness = currFitness
                bestWidth = i
            end
            motifs = []
        end
    end
    
    # return the updatedIndividual and its fitness score
    updatedIndividual = []
    for i in range(1, t)
        push!(updatedIndividual, numericIndividual[i] + bestWidth)
    end
    return updatedIndividual, maxFitness
end

evaluateCandidate (generic function with 1 method)

In [71]:
function AddMutation(m)
    Nuc = ["A", "G", "T", "C"]
    Nmutations = 0
    mutated = m
    # introducing mutations in the motif 
    for j in range(1, Nmutations)
        randomIndex = Int(ceil(rand()*length(m)))
        if randomIndex == 1
            mutated = Nuc[Int(ceil(rand()*4))] * m[2:length(m)]
        elseif randomIndex == length(m)
            mutated = m[1:(length(m)-1)] * Nuc[Int(ceil(rand()*4))]
        else
            mutated = m[1:randomIndex-1] * Nuc[Int(ceil(rand()*4))] * m[randomIndex+1:length(m)]
        end
    end    
    
    return mutated
end


function DnaGenerator(t, m, l)
    DnaSequences = []
    GC = ["G", "C"]
    AT = ["A", "T"]
    lengthExceptMotif = l - length(m)
    baseSequence = ""
    # Generating Dna Sequences, apart from the motif, with 41% GC content
    for i in range(1, ceil(0.41 * lengthExceptMotif))
        baseSequence *= GC[ceil(Int, rand()*2)]
    end
    # Generating Dna Sequences, apart from the motif, with remaining % of AT content
    for i in range(1, lengthExceptMotif - ceil(0.41 * lengthExceptMotif))
        baseSequence *= AT[ceil(Int, rand()*2)]
    end
    
    mutatedMotifs = []
    
    # introducing a different mutation in each motif to be implanted.
    for i in range(1, t)
        push!(mutatedMotifs, AddMutation(m))
    end
    
    #println(mutatedMotifs)
    actualStartPositions = []
    # adding the motif to each randomly generated sequence
    for i in range(1, t)
        permSequence = join(collect(baseSequence)[randperm(length(baseSequence))])

        insertAt = ceil(Int, rand()*(lengthExceptMotif+1))
        push!(actualStartPositions, insertAt)
        if insertAt == 1
            endSequence = mutatedMotifs[i] * permSequence
        elseif insertAt == lengthExceptMotif + 1
            endSequence = permSequence * mutatedMotifs[i]
        else
            endSequence = permSequence[1:insertAt-1] * mutatedMotifs[i] * permSequence[insertAt:lengthExceptMotif]
        end
        push!(DnaSequences, endSequence)
    end
    return DnaSequences, actualStartPositions
end

DnaGenerator (generic function with 1 method)

### Running MDGA
**Inputs**
1. DNA sequences
2. Motif length(width)

**Outputs**
1. Start position
2. Fitness score

In [101]:
function Execute_MDGA(dna_seqs, width)
    # set motif width 
    k = width
    # set sequence length 
    l = length(dna_seqs[1])
    # set number of sequences 
    t = length(dna_seqs)
    # set crossover probability 
    crossProb = 0.4
    # set mutation rate
    mutationRate = 1
    # set mutation probability
    mutationProb = 0.01
    # set shiftWidth 
    shiftWidth = 3
    # repeat experiment g times
    g = 100
    # set population size 
    popSize = 100
    
    # generate DNA sequences with the motif implanted in them at random positions
    DnaSequences = dna_seqs
    #DnaSequences, actualStartPositions = DnaGenerator(t, "ACTGACTGACTGACTGACTG", l)

    # generate first population of individuals - 4 individuals(Test)
    numericIndividuals = []
    regularIndividuals = []

    for i in range(1, popSize)
        # generate a set of random starting points
        numericIndividual = RandomGenerator(l, k, t)
        push!(numericIndividuals, numericIndividual)
        # get the corresponding individual 
        regularIndividual = getIndividual(numericIndividual)
        push!(regularIndividuals, regularIndividual)
    end
    
    #println(numericIndividuals)
    #println(regularIndividuals)
    # evaluate current individuals in population - with shift width and get their best scores
    fitnessScores = []
    for i in range(1, popSize)
        updatedIndividual, fitnessScore = evaluateCandidate(DnaSequences, numericIndividuals[i], shiftWidth, k, t)
        #println(updatedIndividual)
        #println(fitnessScore)
        push!(fitnessScores, fitnessScore)
        numericIndividuals[i] = updatedIndividual
        regularIndividuals[i] = getIndividual(updatedIndividual)
    end
    
    #println(fitnessScores)
    
    # iterations begin    
    for i in range(1, g)
        #print("iteration ")
        #println(i)
        # select 50 parents from the population
        selectedIndices = performSelection(Int(popSize/2), fitnessScores)
        childCount = 0
        # for each pair perform crossover and mutation
        for j in range(1, length(selectedIndices), step=2)
            currCrossProb = rand()
            if currCrossProb <= crossProb

                parent1 = regularIndividuals[selectedIndices[Int(j)]]
                parent2 = regularIndividuals[selectedIndices[Int(j)+1]]
                # perform crossover
                offspring1, offspring2 = performCrossover(parent1, parent2)
                # perform mutation
                mutatedIndividual1 = performMutation(offspring1, mutationRate, mutationProb, k)
                mutatedIndividual2 = performMutation(offspring2, mutationRate, mutationProb, k)  

                if validateIndividual(mutatedIndividual1, DnaSequences, k)
                    mutatedNumericIndividual1 = getNumericIndividual(mutatedIndividual1)
                    # evaluate child1
                    updatedMutatedNumericIndividual1, fitnessScore = evaluateCandidate(DnaSequences, mutatedNumericIndividual1, shiftWidth, k, t)
                    push!(regularIndividuals, mutatedIndividual1)
                    push!(fitnessScores, fitnessScore)
                    push!(numericIndividuals, updatedMutatedNumericIndividual1)
                    childCount += 1
                end
                if validateIndividual(mutatedIndividual2, DnaSequences, k)
                    mutatedNumericIndividual2 = getNumericIndividual(mutatedIndividual2)
                    # evaluate child2
                    updatedMutatedNumericIndividual2, fitnessScore = evaluateCandidate(DnaSequences, mutatedNumericIndividual2, shiftWidth, k, t)
                    push!(regularIndividuals, mutatedIndividual2)
                    push!(fitnessScores, fitnessScore)
                    push!(numericIndividuals, updatedMutatedNumericIndividual2)
                    childCount += 1
                end
            end
        end

        
        #println(childCount)

        # remove the worst performing ones
        sortHelper = sortperm(fitnessScores, rev = true)
        
        fitnessScores = fitnessScores[sortHelper]
        numericIndividuals = numericIndividuals[sortHelper]
        regularIndividuals = regularIndividuals[sortHelper]
        #println(regularIndividuals)
        # change 4 to number of individuals in population
        fitnessScores = fitnessScores[1:popSize]
        numericIndividuals = numericIndividuals[1:popSize]
        regularIndividuals = regularIndividuals[1:popSize]
    end
    
    bestMotifs = []
    
    for i in range(1, length(numericIndividuals[1]))
        push!(bestMotifs, DnaSequences[i][numericIndividuals[1][i]:numericIndividuals[1][i] + k - 1]) 
    end
    bestMotif = consensusMotif(bestMotifs)
    print("consensus Motif ")
    println(bestMotif)
    #print("actual start positions of motif ")
    #println(actualStartPositions)
    print("predicted start positions of motif ")
    println(numericIndividuals[1])
    return numericIndividuals[1], fitnessScores[1]
            
end

Execute_MDGA (generic function with 2 methods)

# Dummy Data

1. Dumma data perfect

In [114]:
# Read a text file
test = open("/Users/sangmi_jeong/Documents/julia/dummy_data_perfect.txt", "r")
lines = readlines(test)
close(test)

l = length(lines)
for i in 1:l
    lines[i] = strip(lines[i])
end

width = parse(Int64, match(r"^(\d+)", lines[1], 1)[1]) # 
t = parse(Int64, match(r"(\d+)$", lines[1], 2)[1])
dna_seqs = lines[2:l];

In [116]:
# Result
best_score = []
best_motifs = []
for i in 1:50
    result = Execute_MDGA(dna_seqs, width)
    push!(best_score, float(result[2]))
    if float(result[2]) == maximum(best_score) 
        best_motifs = result[1]
    end
end

# Data description
println("Motif Width=",width)
println("Number of DNA seqs=",t)
#println("DNA sequences: ")
#for l in 1:L-1
#    println(" ", l,": ", dna_seqs[l])
#end
println("Best Motifs: ")
for l in 1:t
    start_l = Int16(result[1][l])
    println(" ", l,": Start Position = ", start_l)
    println("          Motif  = ", dna_seqs[l][start_l:start_l+width-1])
end

consensus Motif AAAATTTTCAAATTTT
predicted start positions of motif Any[47, 97, 106, 116, 121, 44, 19, 86, 5, 2, 109, 93, 28, 46, 132]
consensus Motif GCTGCAGACAAAATCA
predicted start positions of motif Any[6, 10, 76, 70, 133, 69, 83, 66, 3, 82, 87, 1, 15, 10, 35]
consensus Motif GACTTCAAGTTTAAAG
predicted start positions of motif Any[74, 92, 30, 82, 118, 38, 5, 22, 66, 106, 127, 3, 118, 75, 52]
consensus Motif GTCGATGTCCCAAATC
predicted start positions of motif Any[38, 45, 20, 131, 131, 5, 22, 34, 126, 122, 10, 47, 6, 101, 82]
consensus Motif ATATTTTGTATCAACG
predicted start positions of motif Any[70, 104, 94, 75, 32, 60, 122, 51, 80, 4, 44, 31, 111, 125, 102]
consensus Motif CTTAACTACACTGATG
predicted start positions of motif Any[98, 132, 75, 20, 122, 35, 115, 11, 6, 41, 88, 30, 26, 103, 49]
consensus Motif ACCGACAAAATTTAAA
predicted start positions of motif Any[80, 92, 11, 123, 13, 38, 14, 37, 105, 85, 80, 7, 6, 106, 60]
consensus Motif GTAACGCAAATTGGTA
predicted start positions of 

LoadError: UndefVarError: result not defined

2. Dummy Data Noisy

In [104]:
# Read a text file
test = open("/Users/sangmi_jeong/Documents/julia/dummy_data_noisy.txt", "r")
lines = readlines(test)
close(test)

l = length(lines)
for i in 1:l
    lines[i] = strip(lines[i])
end

width = parse(Int64, match(r"^(\d+)", lines[1], 1)[1]) # 
t = parse(Int64, match(r"(\d+)$", lines[1], 2)[1])
dna_seqs = lines[2:l];

In [111]:
# Result
best_score = []
best_motifs = []
for i in 1:200
    result = Execute_MDGA(dna_seqs, width)
    push!(best_score, float(result[2]))
    if float(result[2]) == maximum(best_score) 
        best_motifs = result[1]
    end
end

# Data description
println("Motif Width=",width)
println("Number of DNA seqs=",t)
#println("DNA sequences: ")
#for l in 1:L-1
#    println(" ", l,": ", dna_seqs[l])
#end
println("Best Motifs: ")
for l in 1:t
    start_l = Int16(result[1][l])
    println(" ", l,": Start Position = ", start_l)
    println("          Motif  = ", dna_seqs[l][start_l:start_l+width-1])
end

consensus Motif CTGAAATTTTAAGA
predicted start positions of motif Any[92, 74, 15, 55, 3, 115, 69, 40, 113, 64, 123, 11, 114, 45, 41]
consensus Motif TAAATAGAGCTCAA
predicted start positions of motif Any[56, 123, 19, 66, 37, 130, 136, 50, 127, 126, 92, 109, 11, 98, 37]
consensus Motif TTAATGGTCTAAGA
predicted start positions of motif Any[109, 42, 27, 47, 34, 32, 114, 60, 47, 11, 136, 54, 70, 52, 106]
consensus Motif ACGACCGAATGATA
predicted start positions of motif Any[50, 32, 63, 38, 67, 70, 65, 97, 12, 87, 80, 56, 31, 16, 9]
consensus Motif TTAAAACGGCGTCA
predicted start positions of motif Any[14, 123, 137, 84, 79, 118, 65, 58, 38, 135, 106, 101, 126, 62, 128]
consensus Motif TTTAGAAATTTGGG
predicted start positions of motif Any[125, 11, 132, 82, 105, 115, 16, 54, 62, 116, 101, 28, 84, 107, 69]
consensus Motif AAGTTTTTAATATT
predicted start positions of motif Any[49, 78, 95, 58, 47, 52, 77, 105, 33, 29, 36, 8, 131, 101, 58]
consensus Motif TGCCAATTGTAAAA
predicted start positions of m

consensus Motif ACTTAGGTTGCGAA
predicted start positions of motif Any[84, 5, 74, 22, 47, 122, 14, 39, 23, 57, 24, 16, 136, 25, 74]
consensus Motif AAATTCTCAAATTT
predicted start positions of motif Any[58, 98, 3, 50, 41, 76, 51, 14, 50, 3, 46, 50, 89, 87, 66]
consensus Motif TAAAAACTTAAAGG
predicted start positions of motif Any[102, 63, 42, 103, 122, 137, 131, 102, 10, 77, 61, 92, 79, 90, 63]
consensus Motif AAACTTTCCAAGTA
predicted start positions of motif Any[95, 126, 104, 86, 28, 44, 124, 134, 43, 54, 57, 119, 122, 59, 132]
consensus Motif CTTGTTCAACGAAA
predicted start positions of motif Any[79, 55, 121, 84, 9, 27, 114, 30, 2, 111, 107, 17, 16, 54, 113]
consensus Motif AAATTTTAAAATCT
predicted start positions of motif Any[58, 29, 119, 116, 132, 111, 97, 89, 11, 28, 22, 102, 29, 48, 65]
consensus Motif AATTTTCTAAACTT
predicted start positions of motif Any[97, 45, 96, 59, 108, 112, 133, 70, 108, 46, 119, 85, 49, 49, 65]
consensus Motif TTTGAGAATATGAA
predicted start positions of motif

consensus Motif TGAAAATTTCCGAA
predicted start positions of motif Any[94, 34, 116, 47, 81, 117, 61, 74, 18, 4, 25, 12, 20, 63, 87]
consensus Motif CTCAGTTGTTTGAT
predicted start positions of motif Any[100, 3, 28, 54, 53, 51, 32, 101, 30, 66, 81, 13, 63, 63, 99]
consensus Motif ACGTGCAAAAAATT
predicted start positions of motif Any[51, 39, 125, 58, 20, 38, 15, 36, 26, 7, 123, 57, 100, 91, 26]
consensus Motif AAGACTCAGGGCTT
predicted start positions of motif Any[87, 26, 103, 116, 13, 65, 10, 64, 41, 37, 133, 101, 95, 48, 5]
consensus Motif TCAACGAAATTTAG
predicted start positions of motif Any[118, 60, 75, 129, 14, 99, 135, 129, 92, 104, 101, 12, 69, 17, 77]
consensus Motif AAGATTCAAAAACT
predicted start positions of motif Any[49, 97, 80, 65, 108, 53, 97, 33, 129, 19, 37, 71, 131, 21, 134]
consensus Motif TCTCGCAAATCTTT
predicted start positions of motif Any[99, 112, 70, 63, 94, 113, 99, 45, 25, 84, 121, 124, 2, 93, 21]
consensus Motif TAAAAATTCTCGGA
predicted start positions of motif Any[

consensus Motif TTGTCCAGCCGCAA
predicted start positions of motif Any[116, 49, 101, 39, 31, 65, 82, 124, 59, 115, 9, 51, 54, 62, 120]
consensus Motif AAAATTTTCAATTC
predicted start positions of motif Any[95, 127, 58, 65, 77, 62, 104, 104, 42, 132, 45, 66, 29, 47, 65]
consensus Motif CCAAAATTTTTAAA
predicted start positions of motif Any[29, 121, 41, 56, 71, 50, 14, 100, 92, 128, 32, 12, 77, 77, 116]
consensus Motif TTTAAAGGCTGAAA
predicted start positions of motif Any[82, 70, 118, 84, 119, 115, 76, 3, 22, 33, 23, 92, 19, 28, 55]
consensus Motif GCAACGCCTACAGT
predicted start positions of motif Any[28, 125, 74, 24, 134, 126, 132, 13, 105, 36, 77, 100, 79, 103, 71]
consensus Motif TGGGATTTATAATG
predicted start positions of motif Any[47, 64, 68, 112, 59, 125, 42, 73, 72, 98, 85, 72, 28, 71, 101]
consensus Motif CTCTAAATTTGCAA
predicted start positions of motif Any[84, 120, 124, 46, 78, 136, 136, 39, 38, 136, 134, 17, 51, 87, 95]
consensus Motif CACCCGCGTAAATT
predicted start positions of 

LoadError: UndefVarError: result not defined

# Simulation

**Inputs**
1. DNA sequences
2. actual_motifs
3. motif length(width)
4. number of repeats


**Outputs**
1. Start positions
2. Predicted Motifs
3. Time
4. Fitness score
5. Occurance of motifs(%)
6. Accuracy(%)

In [106]:
function simul(dna_seqs, actual_motifs, motif_pos, width, rep) 
    # dna_seqs = array of DNA seqs
    # actual_motifs = arrayf of actual motifs
    # motif_pos = array of position of actual motifs in integer
    t = length(dna_seqs)
    l = length(dna_seqs[1])
    
    start_pos = fill([],rep)
    pred_motifs = fill([],rep)
    
    running_time = []
    fitness_score = []
    occurance_motifs = []
    sensitivity = []
    specificity = []
    
    for i in 1:rep
        println("repeat ", i)
        result = @timed Execute_MDGA(dna_seqs, width)

        # 3. Time
        push!(running_time, result.time)
        
        result = result[1]
        # 4. Fitness Score
        push!(fitness_score, float(result[2]))
        
        # Prediction
        # 1. and 2.
        start_pos[i] = result[1]
        pred_motif = []
        for j in 1:t
            start_j = result[1][j]
            push!(pred_motif, dna_seqs[j][start_j:start_j+width-1])
        end
        pred_motifs[i] = pred_motif
        
        # 4. Occurance of motifs(%)
        push!(occurance_motifs, sum([pred_motif[q] == actual_motifs[q] for q in 1:t])/t)
        
        # 5. Accuracy(%)
        nTP = Array{Int64}(undef, t);
        nFP = Array{Int64}(undef, t);
        nFN = Array{Int64}(undef, t);
        nTN = Array{Int64}(undef, t);
        for j in 1:t
            if abs(result[1][j] - motif_pos[j]) > width
                nTP[j] = 0
                nFN[j] = width
                nFP[j] = width
                nTN[j] = l - 2*width
            else
                if result[1][j] <= motif_pos[j]
                    nTP[j] = abs(result[1][j]+width - motif_pos[j])
                else
                    nTP[j] = abs(motif_pos[j]+width - result[1][j])
                end
                nFN[j] = width - nTP[j]
                nFP[j] = width - nTP[j]
                nTN[j] = l - nTP[j] - 2*nFN[j]
            end
        end

        push!(sensitivity, sum(nTP)/(sum(nTP)+sum(nFN)))
        push!(specificity, sum(nTN)/(sum(nTN)+sum(nFP)))
        
    end
    return start_pos, pred_motifs, running_time, fitness_score, occurance_motifs, sensitivity, specificity
end

simul (generic function with 1 method)

### Read files

In [107]:
function read_files(test)
   # Read DNA Sequences: seqs
    filename = "/Users/sangmi_jeong/Documents/julia/" * test * ".fasta"
    file = open(filename, "r")
    lines = readlines(file)
    close(file)

    L = 2
    seqs = []
    while L < length(lines) + 1
        push!(seqs, lines[L])
        L = L+2
    end

    # Read Actual Motifs : actual_motifs
    file_motifs = "/Users/sangmi_jeong/Documents/julia/" * test * "_motifs.txt"
    file = open(file_motifs, "r")
    actual_motifs = readlines(file)
    close(file)

    # Read Motif positions: motif_pos
    file_pos = "/Users/sangmi_jeong/Documents/julia/" * test * "_pos.txt"
    file = open(file_pos, "r")
    motif_pos = readlines(file)
    close(file)
    motif_pos = [parse(Int64, pos) for pos in motif_pos]
    
    return seqs, actual_motifs, motif_pos
end

read_files (generic function with 1 method)

### TEST 1 
t = 50 --------------------# num of DNA seqs  <br>
k = 14 ---------------------# Motif length     <br>
m = consensus_motif(k) -----# consensus motif(implanted)  <br>
l = 200 --------------------# DNA length       <br>
mut = 2 --------------------# num of mutations <br> 
filename = "test1" ---------# file in fasta format <br>

In [108]:
test = "test1"
width = 14
rep = 100

files = read_files(test)
seqs = files[1]
actual_motifs = files[2]
motif_pos = files[3]

#simul_result1 = simul(seqs, actual_motifs, motif_pos, width, rep)

20-element Vector{Int64}:
 23
 38
 85
 35
 27
 10
 56
 71
 66
 33
  7
  8
 40
 83
 38
 75
 59
 15
 31
 64

In [109]:
width=14
rep=10
simul_result1 = simul(seqs, actual_motifs, motif_pos, width, rep);

repeat 1
consensus Motif TGCACGGTATACAC
predicted start positions of motif Any[40, 40, 12, 28, 75, 12, 26, 24, 59, 4, 86, 10, 58, 66, 27, 25, 24, 67, 57, 44]
repeat 2
consensus Motif ATTCAACACGTAAT
predicted start positions of motif Any[7, 79, 83, 37, 21, 33, 77, 45, 64, 22, 9, 84, 7, 81, 23, 21, 57, 53, 16, 26]
repeat 3
consensus Motif AAGGATTAGTAACT
predicted start positions of motif Any[41, 33, 35, 40, 22, 34, 30, 76, 13, 48, 16, 17, 62, 29, 18, 86, 33, 68, 11, 79]
repeat 4
consensus Motif TTGATTAAAACGGT
predicted start positions of motif Any[4, 29, 53, 17, 23, 83, 81, 67, 3, 30, 3, 50, 41, 45, 3, 18, 34, 55, 4, 12]
repeat 5


LoadError: InterruptException:

In [110]:
function plot(simul_result)
    start_pos = simul_result[1]
    motifs = simul_result1[2]
    time = simul_result1[3]
    sensitivity = simul_result1[4]
    specificity = simul_result1[5]
    

#println(start_pos)
#println(motifs)
#println(time)
#println(sensitivity)
#println(specificity)

LoadError: syntax: incomplete: "function" at In[110]:1 requires end

In [51]:
x = simul_result1[4]
y = ones(length(simul_result1[5])) - simul_result1[5]
df = DataFrame(x=[0;x;1], y=[0;y;1])

sort!(df, [:y])


,x,y
,Any,Float64
1,0,0.0
2,2.69126,0.95
3,2.67944,1.0
4,2.66765,1.0
5,2.66761,1.0
6,2.85126,1.0
7,2.6914,1.0
8,2.79425,1.0
9,2.92917,1.0


In [33]:
#gr()
plot1 = plot(df[:x], df[:y], color=:green, label="ROC curve", 
    ylabel="sensitivity(TruePositiveRate)",
    left_margin = 5Plots.mm, right_margin = 15Plots.mm)
xlabel!("1-specificity(FalsePositiveRate)")
display(plot1)
#savefig("ROCcurve.pdf")

LoadError: UndefVarError: df not defined